# Chapter 15. Automating Everyday Tasks

Companion notebook for *Python from Zero*, Chapter 15. Run the cells from
top to bottom with **Shift + Enter**. Everything here works on a sample
folder the notebook creates, `notebooks/output/pet_photos`, never on your
own files. To start again from scratch, delete the `pet_photos` and
`pet_photos_backup` folders inside `notebooks/output` and rerun the
notebook.

## 15.2 A safe place to practice

In [1]:
from pathlib import Path

photos = Path("output/pet_photos")
photos.mkdir(parents=True, exist_ok=True)

sample_files = [
    "IMG_20260412_093000.jpg",
    "IMG_20260412_093512.JPG",
    "IMG_20260503_141020.jpeg",
    "IMG_20260521_160245.jpg",
    "IMG_20260603_101500.jpg",
    "Peanut at the park.jpg",
    "Truffle SLEEPING.PNG",
    "Kiwi bath time.MOV",
    "adoption form Gus.pdf",
    "vet notes Mochi.txt",
]
for name in sample_files:
    path = photos / name
    path.write_text(f"Pretend file: {name}\n", encoding="utf-8")

print(f"Created {len(sample_files)} files in {photos}")

Created 10 files in output/pet_photos


## 15.3 Looking inside a folder

In [2]:
for path in sorted(photos.iterdir()):
    print(path.name)

IMG_20260412_093000.jpg
IMG_20260412_093512.JPG
IMG_20260503_141020.jpeg
IMG_20260521_160245.jpg
IMG_20260603_101500.jpg
Kiwi bath time.MOV
Peanut at the park.jpg
Truffle SLEEPING.PNG
adoption form Gus.pdf
vet notes Mochi.txt


In [3]:
for path in sorted(photos.iterdir()):
    size = path.stat().st_size
    print(f"{path.suffix.lower():<6}{size:>4} bytes  {path.name}")

.jpg    38 bytes  IMG_20260412_093000.jpg
.jpg    38 bytes  IMG_20260412_093512.JPG
.jpeg   39 bytes  IMG_20260503_141020.jpeg
.jpg    38 bytes  IMG_20260521_160245.jpg
.jpg    38 bytes  IMG_20260603_101500.jpg
.mov    33 bytes  Kiwi bath time.MOV
.jpg    37 bytes  Peanut at the park.jpg
.png    35 bytes  Truffle SLEEPING.PNG
.pdf    36 bytes  adoption form Gus.pdf
.txt    34 bytes  vet notes Mochi.txt


In [4]:
counts = {}
for path in photos.iterdir():
    if path.is_file():
        kind = path.suffix.lower()
        counts[kind] = counts.get(kind, 0) + 1

for kind in sorted(counts):
    print(f"{kind:<6}{counts[kind]}")

.jpeg 1
.jpg  5
.mov  1
.pdf  1
.png  1
.txt  1


In [5]:
for path in sorted(photos.glob("IMG_*")):
    print(path.name)

IMG_20260412_093000.jpg
IMG_20260412_093512.JPG
IMG_20260503_141020.jpeg
IMG_20260521_160245.jpg
IMG_20260603_101500.jpg


In [6]:
from datetime import datetime

path = photos / "Peanut at the park.jpg"
modified = datetime.fromtimestamp(path.stat().st_mtime)
print(modified.strftime("%Y-%m-%d"))

2026-10-06


## 15.4 Copying, renaming and moving

In [7]:
import shutil

backup = Path("output/pet_photos_backup")
shutil.copytree(photos, backup, dirs_exist_ok=True)

print(len(list(backup.iterdir())), "files in", backup)

10 files in output/pet_photos_backup


In [8]:
website = Path("output/website")
website.mkdir(exist_ok=True)
shutil.copy2(photos / "Peanut at the park.jpg", website / "peanut.jpg")

print((photos / "Peanut at the park.jpg").exists())
print((website / "peanut.jpg").exists())

True
True


In [9]:
old_path = backup / "vet notes Mochi.txt"
new_path = old_path.with_name("mochi-vet-notes.txt")
print(new_path)

old_path.rename(new_path)
print(old_path.exists(), new_path.exists())

output/pet_photos_backup/mochi-vet-notes.txt
False True


In [10]:
videos = backup / "videos"
videos.mkdir(exist_ok=True)
shutil.move(backup / "Kiwi bath time.MOV", videos / "Kiwi bath time.MOV")

for path in sorted(videos.iterdir()):
    print(path)

output/pet_photos_backup/videos/Kiwi bath time.MOV


## 15.5 Batch renaming, with a dry run first

In [11]:
PHOTO_TYPES = [".jpg", ".jpeg", ".png"]


def tidy_name(path):
    """Return the tidy file name for one photo."""
    stem = path.stem.lower().replace(" ", "-")
    if stem.startswith("img_") and len(stem) == 19:
        # A camera name like img_20260412_093000 becomes 2026-04-12_093000
        day = stem[4:12]
        stem = f"{day[:4]}-{day[4:6]}-{day[6:]}_{stem[13:]}"
    suffix = path.suffix.lower()
    if suffix == ".jpeg":
        suffix = ".jpg"
    return stem + suffix


def plan_renames(folder):
    """Return a list of (old_path, new_path) pairs for the photos."""
    plan = []
    for path in sorted(folder.iterdir()):
        if path.is_file() and path.suffix.lower() in PHOTO_TYPES:
            new_path = path.with_name(tidy_name(path))
            if new_path != path:
                plan.append((path, new_path))
    return plan

In [12]:
for name in ["IMG_20260412_093512.JPG", "IMG_20260503_141020.jpeg",
             "Peanut at the park.jpg", "Truffle SLEEPING.PNG"]:
    print(f"{name:<26}->  {tidy_name(Path(name))}")

IMG_20260412_093512.JPG   ->  2026-04-12_093512.jpg
IMG_20260503_141020.jpeg  ->  2026-05-03_141020.jpg
Peanut at the park.jpg    ->  peanut-at-the-park.jpg
Truffle SLEEPING.PNG      ->  truffle-sleeping.png


In [13]:
def rename_photos(folder, dry_run=True):
    """Tidy every photo name in folder. Only prints if dry_run is True."""
    for path in sorted(folder.iterdir()):
        if not path.is_file() or path.suffix.lower() not in PHOTO_TYPES:
            continue
        new_path = path.with_name(tidy_name(path))
        if new_path == path:
            continue
        if new_path.exists():
            print(f"SKIP  {path.name} ({new_path.name} already exists)")
        elif dry_run:
            print(f"WOULD RENAME  {path.name}  ->  {new_path.name}")
        else:
            path.rename(new_path)
            print(f"RENAMED  {path.name}  ->  {new_path.name}")


rename_photos(photos)

WOULD RENAME  IMG_20260412_093000.jpg  ->  2026-04-12_093000.jpg
WOULD RENAME  IMG_20260412_093512.JPG  ->  2026-04-12_093512.jpg
WOULD RENAME  IMG_20260503_141020.jpeg  ->  2026-05-03_141020.jpg
WOULD RENAME  IMG_20260521_160245.jpg  ->  2026-05-21_160245.jpg
WOULD RENAME  IMG_20260603_101500.jpg  ->  2026-06-03_101500.jpg
WOULD RENAME  Peanut at the park.jpg  ->  peanut-at-the-park.jpg
WOULD RENAME  Truffle SLEEPING.PNG  ->  truffle-sleeping.png


In [14]:
rename_photos(photos, dry_run=False)

RENAMED  IMG_20260412_093000.jpg  ->  2026-04-12_093000.jpg
RENAMED  IMG_20260412_093512.JPG  ->  2026-04-12_093512.jpg
RENAMED  IMG_20260503_141020.jpeg  ->  2026-05-03_141020.jpg
RENAMED  IMG_20260521_160245.jpg  ->  2026-05-21_160245.jpg
RENAMED  IMG_20260603_101500.jpg  ->  2026-06-03_101500.jpg
RENAMED  Peanut at the park.jpg  ->  peanut-at-the-park.jpg
RENAMED  Truffle SLEEPING.PNG  ->  truffle-sleeping.png


In [15]:
rename_photos(photos, dry_run=False)
print("Done.")

Done.


## 15.6 Organizing files into folders

In [16]:
def show_tree(folder):
    """Print a folder's contents, one level of subfolders deep."""
    print(f"{folder.name}/")
    for path in sorted(folder.iterdir()):
        if path.is_dir():
            print(f"    {path.name}/")
            for child in sorted(path.iterdir()):
                print(f"        {child.name}")
        else:
            print(f"    {path.name}")


show_tree(photos)

pet_photos/
    2026-04-12_093000.jpg
    2026-04-12_093512.jpg
    2026-05-03_141020.jpg
    2026-05-21_160245.jpg
    2026-06-03_101500.jpg
    Kiwi bath time.MOV
    adoption form Gus.pdf
    peanut-at-the-park.jpg
    truffle-sleeping.png
    vet notes Mochi.txt


In [17]:
FOLDER_FOR_TYPE = {
    ".jpg": "photos", ".jpeg": "photos", ".png": "photos",
    ".mov": "videos", ".mp4": "videos",
    ".pdf": "documents", ".txt": "documents", ".docx": "documents",
}


def organize_by_type(folder, dry_run=True):
    """Move each file in folder into a subfolder for its type."""
    for path in sorted(folder.iterdir()):
        if not path.is_file():
            continue
        subfolder = folder / FOLDER_FOR_TYPE.get(path.suffix.lower(),
                                                 "other")
        target = subfolder / path.name
        if target.exists():
            print(f"SKIP  {path.name} (already in {subfolder.name}/)")
        elif dry_run:
            print(f"WOULD MOVE  {path.name}  ->  {subfolder.name}/")
        else:
            subfolder.mkdir(exist_ok=True)
            shutil.move(path, target)


organize_by_type(photos)

WOULD MOVE  2026-04-12_093000.jpg  ->  photos/
WOULD MOVE  2026-04-12_093512.jpg  ->  photos/
WOULD MOVE  2026-05-03_141020.jpg  ->  photos/
WOULD MOVE  2026-05-21_160245.jpg  ->  photos/
WOULD MOVE  2026-06-03_101500.jpg  ->  photos/
WOULD MOVE  Kiwi bath time.MOV  ->  videos/
WOULD MOVE  adoption form Gus.pdf  ->  documents/
WOULD MOVE  peanut-at-the-park.jpg  ->  photos/
WOULD MOVE  truffle-sleeping.png  ->  photos/
WOULD MOVE  vet notes Mochi.txt  ->  documents/


In [18]:
organize_by_type(photos, dry_run=False)
show_tree(photos)

pet_photos/
    documents/
        adoption form Gus.pdf
        vet notes Mochi.txt
    photos/
        2026-04-12_093000.jpg
        2026-04-12_093512.jpg
        2026-05-03_141020.jpg
        2026-05-21_160245.jpg
        2026-06-03_101500.jpg
        peanut-at-the-park.jpg
        truffle-sleeping.png
    videos/
        Kiwi bath time.MOV


In [19]:
def month_folder_name(path):
    """'2026-04' for a name like 2026-04-12_093000.jpg, else 'undated'."""
    name = path.name
    if name.startswith("20") and name[4] == "-" and name[7] == "-":
        return name[:7]
    return "undated"


photo_folder = photos / "photos"
for path in sorted(photo_folder.iterdir()):
    if path.is_file():
        month = photo_folder / month_folder_name(path)
        if (month / path.name).exists():
            print(f"SKIP  {path.name}")
            continue
        month.mkdir(exist_ok=True)
        shutil.move(path, month / path.name)

show_tree(photo_folder)

photos/
    2026-04/
        2026-04-12_093000.jpg
        2026-04-12_093512.jpg
    2026-05/
        2026-05-03_141020.jpg
        2026-05-21_160245.jpg
    2026-06/
        2026-06-03_101500.jpg
    undated/
        peanut-at-the-park.jpg
        truffle-sleeping.png


## 15.7 A weekly report from the shelter's files

In [20]:
import csv
from datetime import date, timedelta

DATA = Path("../data")


def read_csv(path):
    """Return the rows of a CSV file as a list of dictionaries."""
    with open(path, encoding="utf-8", newline="") as file:
        return list(csv.DictReader(file))


def in_period(text, start, end):
    """True if a date written like "2026-06-03" is from start to end."""
    if text == "":
        return False
    return start <= date.fromisoformat(text) <= end


def build_report(start, days):
    """Return the lines of the report for the given period."""
    end = start + timedelta(days=days - 1)
    donations = [row for row in read_csv(DATA / "donations.csv")
                 if in_period(row["date"], start, end)]
    animals = read_csv(DATA / "animals.csv")
    arrived = [row for row in animals if in_period(row["arrived"], start, end)]
    adopted = [row for row in animals
               if in_period(row["adopted_on"], start, end)]
    visits = 0
    with open(DATA / "visitor_log.txt", encoding="utf-8") as file:
        for line in file:
            if line.strip() and in_period(line.split()[0], start, end):
                visits += 1

    first_day = start.strftime("%a %d %b %Y")
    last_day = end.strftime("%a %d %b %Y")
    lines = ["Sunny Paws weekly report", f"{first_day} to {last_day}",
             "=" * 40, ""]
    total = 0
    biggest = None
    for row in donations:
        amount = int(row["amount"])
        total += amount
        if biggest is None or amount > int(biggest["amount"]):
            biggest = row
    lines.append(f"Donations: {len(donations)}, ${total:,} in total")
    if biggest is not None:
        lines.append(f"  Biggest: ${biggest['amount']} from "
                     f"{biggest['donor']} on {biggest['date']}")
    lines.append("")
    lines.append(f"Arrived: {len(arrived)}")
    for row in arrived:
        lines.append(f"  {row['id']}  {row['name']} ({row['species']})")
    lines.append(f"Adopted: {len(adopted)}")
    for row in adopted:
        lines.append(f"  {row['id']}  {row['name']} ({row['species']})")
    lines.append("")
    lines.append(f"Visits in the visitor log: {visits}")
    return lines

In [21]:
for line in build_report(date(2026, 6, 1), 7):
    print(line)

Sunny Paws weekly report
Mon 01 Jun 2026 to Sun 07 Jun 2026

Donations: 5, $235 in total
  Biggest: $150 from Maple Falls Rotary Club on 2026-06-06

Arrived: 4
  SP-045  Pip (Guinea pig)
  SP-046  Clover (Rabbit)
  SP-047  Nugget (Dog)
  SP-048  Nova (Dog)
Adopted: 2
  SP-033  Sprout (Dog)
  SP-035  Echo (Cat)

Visits in the visitor log: 14


## 15.8 Command-line arguments

In [22]:
import argparse

parser = argparse.ArgumentParser(
    description="Print the Sunny Paws weekly report.")
parser.add_argument("start",
                    help='first day, like 2026-06-01, or "last" for '
                         "last week")
parser.add_argument("--days", type=int, default=7,
                    help="how many days to cover (default: 7)")
parser.add_argument("--save", action="store_true",
                    help="also save the report in the output folder")

args = parser.parse_args(["2026-06-01", "--days", "3"])
print(args)
print(args.start, args.days, args.save)

Namespace(start='2026-06-01', days=3, save=False)
2026-06-01 3 False


## 15.10 Common mistakes

In [23]:
shutil.copytree(photos, backup)

FileExistsError: [Errno 17] File exists: 'output/pet_photos_backup'

In [24]:
shutil.copy2(website / "peanut.jpg", website / "peanut-copy.jpg")
shutil.move(website / "peanut-copy.jpg", website / "archive")

for path in sorted(website.iterdir()):
    if path.is_dir():
        print(path.name, "is a folder")
    else:
        print(path.name, "is a file")

archive is a file
peanut.jpg is a file


## Exercises

Solutions are in Appendix D of the book.

1. Print the number of files and their total size in bytes in the sample `pet_photos` folder, including the files inside all its subfolders. *Hint:* `folder.rglob("*")` works like `glob("*")` but also looks inside every subfolder, however deep.
2. Write a function `preview_names(folder)` that prints, for *every* file in a folder (not only photos), its name and the name `tidy_name()` would give it. Which names would change apart from the photos?
3. Add a `--prefix` option to `ch15_rename_photos.py`, so that `--prefix sunny-paws` puts `sunny-paws-` at the start of every new photo name. Without `--prefix`, the program should behave exactly as before.
4. Write a program `ch15_welcome.py` that takes an animal's name and species from the command line with `argparse` and prints `Welcome to Sunny Paws, Biscuit the rabbit!`. If the species is not one the shelter takes (dog, cat, rabbit or guinea pig, in any mix of capitals), stop with `parser.error()` and a friendly message.
5. Add a `--method` option to the weekly report that limits the donation figures to one payment method. Without it, the report counts every donation as before. *Hint:* `add_argument("--method", choices=["cash", "card", "online"])` makes `argparse` accept only those three values.
6. Write a program that makes a dated backup of a folder: `output/pet_photos` copied to `output/backups/pet_photos_YYYY-MM-DD`, using today's date. It must refuse, with a message, if a backup with that name already exists.
7. Write a program that takes a folder as a command-line argument and prints the newest and the oldest file in it by modification time (`st_mtime`). Try it on `../data`.